Base learners:

- rf_custom_F1
- rf_custom_Recall
- AvalancheNet (neural network)

Implements a stacking ensemble using two custom Random Forest classifiers and a PyTorch neural network (AvalancheNet) as base models. A meta-learner (Logistic Regression or Gradient Boosting) is trained on out-of-sample predictions.


In [2]:
# Core packages
import os  # for file and path operations
import warnings  # to suppress warnings
warnings.filterwarnings("ignore")

# Data manipulation
import pandas as pd  # for handling dataframes
import numpy as np  # for numerical operations

# Visualization
import matplotlib.pyplot as plt  # for general plotting
import seaborn as sns  # for enhanced statistical plotting
from matplotlib.pylab import rcParams  # for customizing plot size and appearance

# Machine learning - preprocessing & model evaluation
from sklearn.model_selection import train_test_split, cross_val_predict, cross_val_score  # data splitting and cross-validation
from sklearn.preprocessing import StandardScaler  # feature standardization

# Machine learning - models
from sklearn.tree import DecisionTreeClassifier, plot_tree  # decision tree model and plotting
from sklearn.ensemble import RandomForestClassifier  # random forest model

# Machine learning - metrics
from sklearn.metrics import (
    ConfusionMatrixDisplay,  # plot confusion matrix
    classification_report,  # detailed classification report
    confusion_matrix,  # confusion matrix array
    accuracy_score,  # accuracy metric
    precision_score,  # precision metric
    recall_score,  # recall metric
    f1_score  # F1 score
)

# PyTorch - deep learning
import torch  # base PyTorch package
import torch.nn as nn  # neural network layers
import torch.optim as optim  # optimizers (e.g., Adam, SGD)

from sklearn.base import BaseEstimator, ClassifierMixin

import seaborn as sns


# Split the data correctly

## Splitting Strategy

To avoid data leakage and build a robust ensemble:
- **Base Set (60%)**: Train base learners (RFs and NN)
- **Meta Set (20%)**: Generate predictions from base learners and train meta-learner
- **Test Set (20%)**: Evaluate the full stacked model (never seen during training)

This 3-part split is essential to ensure generalization.

In [3]:
# Load and clean data
full = pd.read_csv("C:/Users/annaw/Desktop/DataScience/Datasets/full_dataset.csv", dtype={91: str})
print(f"X_train shape: {full.shape}")

# drop columns with >20% missing
full_clean = full.dropna(thresh=len(full)*0.8, axis=1)  
print(full_clean['avalancheDay1'].value_counts())

full_clean = full_clean.dropna()
full_clean = full_clean.select_dtypes(include=[np.number])
full_clean = full_clean.drop(columns=[col for col in ['year', 'datum'] if col in full_clean.columns])
print(full_clean['avalancheDay1'].value_counts())

# Split features and target
X = full_clean.drop(columns=["avalancheDay1"])
y = full_clean["avalancheDay1"]


X_train shape: (11362, 95)
avalancheDay1
0    10638
1      724
Name: count, dtype: int64
avalancheDay1
0    9598
1     695
Name: count, dtype: int64


In [4]:
#I need to split the data carefully and save some testing data for the stacking model 

# 1) Use one dataset to train the BASE models (X_train_base, y_train_base)
# 2) Use another dataset to train the meta-model on base model predictions (X_train_meta, y_train_meta)
# 3) Usa a third, untouched dataset to test the full stacked model (X_test_meta, y_test_meta)

# Split full data into train+meta and final test
X_temp, X_test_meta, y_temp, y_test_meta = train_test_split(X, y, test_size=0.2, random_state=42)

# Now split train+meta into base and meta sets (e.g. 60% base, 20% meta)
X_train_base, X_train_meta, y_train_base, y_train_meta = train_test_split(X_temp, y_temp, test_size=0.25, random_state=42)  # 0.25 * 0.8 = 0.2

# Standardize
scaler = StandardScaler()
X_train_base_scaled = scaler.fit_transform(X_train_base)
X_train_meta_scaled = scaler.transform(X_train_meta)
X_test_meta_scaled = scaler.transform(X_test_meta)

# BASE MODELS

In [5]:
class TorchNNWrapper(BaseEstimator, ClassifierMixin):
    def __init__(self, input_dim, threshold=0.5, epochs=100, lr=0.001, pos_weight=5.0):
        self.threshold = threshold
        self.epochs = epochs
        self.lr = lr
        self.pos_weight = pos_weight  
        self.input_dim = input_dim

    def _build_model(self):
        model = nn.Sequential(
            nn.Linear(self.input_dim, 64),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(32, 1)
        )
        return model

    def fit(self, X, y):
        X_tensor = torch.tensor(X, dtype=torch.float32)
        y_tensor = torch.tensor(y.values, dtype=torch.float32).unsqueeze(1)

        self.model = self._build_model()
        optimizer = optim.Adam(self.model.parameters(), lr=self.lr)
        criterion = nn.BCEWithLogitsLoss(pos_weight=torch.tensor([self.pos_weight]))

        for _ in range(self.epochs):
            self.model.train()
            optimizer.zero_grad()
            outputs = self.model(X_tensor)
            loss = criterion(outputs, y_tensor)
            loss.backward()
            optimizer.step()

        return self

    def predict_proba(self, X):
        self.model.eval()
        with torch.no_grad():
            X_tensor = torch.tensor(X, dtype=torch.float32)
            logits = self.model(X_tensor)
            probs = torch.sigmoid(logits).numpy().flatten()
        return np.vstack([1 - probs, probs]).T

    def predict(self, X):
        return (self.predict_proba(X)[:, 1] > self.threshold).astype(int)

In [7]:
# F1-optimized RF (train on base set)
rf_custom_F1 = RandomForestClassifier(
    class_weight=None,
    max_depth=20,
    max_features=None,
    min_samples_leaf=5,
    n_estimators=100,
    random_state=42,
    n_jobs=-1
)

# Recall-optimized RF (train on base set)
rf_custom_Recall = RandomForestClassifier(
    class_weight='balanced',
    max_depth=10,
    max_features="sqrt",
    min_samples_leaf=200,
    n_estimators=100,
    random_state=42,
    n_jobs=-1
)

# STACKING (META-MODEL)

In [6]:
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import classification_report, confusion_matrix

# 1. Train base models on base set
rf_custom_F1.fit(X_train_base_scaled, y_train_base)
rf_custom_Recall.fit(X_train_base_scaled, y_train_base)
nn_wrapper = TorchNNWrapper(input_dim=X_train_base_scaled.shape[1], threshold=0.5, epochs=200)
nn_wrapper.fit(X_train_base_scaled, y_train_base)

# 2. Predict meta-set using pretrained base models
meta_features = np.column_stack([
    rf_custom_F1.predict_proba(X_train_meta_scaled)[:, 1],
    rf_custom_Recall.predict_proba(X_train_meta_scaled)[:, 1],
    nn_wrapper.predict_proba(X_train_meta_scaled)[:, 1]
])

# Train meta-learner
meta_learner = GradientBoostingClassifier(n_estimators=50, learning_rate=0.1, max_depth=3)
meta_learner.fit(meta_features, y_train_meta)

# 4. Evaluate on final test set
test_features = np.column_stack([
    rf_custom_F1.predict_proba(X_test_meta_scaled)[:, 1],
    rf_custom_Recall.predict_proba(X_test_meta_scaled)[:, 1],
    nn_wrapper.predict_proba(X_test_meta_scaled)[:, 1]
])

y_pred_stack = meta_learner.predict(test_features)

# Evaluate on meta test set
print("\n=== Stacked Model Performance ===")
print(classification_report(y_test_meta, y_pred_stack))
print(confusion_matrix(y_test_meta, y_pred_stack))


=== Stacked Model Performance ===
              precision    recall  f1-score   support

           0       0.99      1.00      0.99      1923
           1       0.97      0.88      0.92       136

    accuracy                           0.99      2059
   macro avg       0.98      0.94      0.96      2059
weighted avg       0.99      0.99      0.99      2059

[[1919    4]
 [  17  119]]


In [12]:
from sklearn.linear_model import LogisticRegressionCV
from sklearn.metrics import classification_report, confusion_matrix, precision_recall_curve

# 1. Train base models on base set
rf_custom_F1.fit(X_train_base_scaled, y_train_base)
rf_custom_Recall.fit(X_train_base_scaled, y_train_base)
nn_wrapper = TorchNNWrapper(input_dim=X_train_base_scaled.shape[1], threshold=0.5, epochs=200)
nn_wrapper.fit(X_train_base_scaled, y_train_base)

# 2. Predict meta-set using pretrained base models
meta_features = np.column_stack([
    rf_custom_F1.predict_proba(X_train_meta_scaled)[:, 1],
    rf_custom_Recall.predict_proba(X_train_meta_scaled)[:, 1],
    nn_wrapper.predict_proba(X_train_meta_scaled)[:, 1]
])

meta_scaler = StandardScaler()
meta_features_scaled = meta_scaler.fit_transform(meta_features)

# Train meta-learner
meta_learner = LogisticRegressionCV(cv=5, max_iter=1000, scoring='f1')
meta_learner.fit(meta_features, y_train_meta)

# 4. Evaluate on final test set
test_features = np.column_stack([
    rf_custom_F1.predict_proba(X_test_meta_scaled)[:, 1],
    rf_custom_Recall.predict_proba(X_test_meta_scaled)[:, 1],
    nn_wrapper.predict_proba(X_test_meta_scaled)[:, 1]
])

y_pred_stack = meta_learner.predict(test_features)

# Evaluate on meta test set
print("\n=== Logistic Regression Meta Model ===")
print(classification_report(y_test_meta, y_pred_stack))

# Evaluate
cm_staked = confusion_matrix(y_test_meta, y_pred_stack)

print(cm_staked)


=== Logistic Regression Meta Model ===
              precision    recall  f1-score   support

           0       0.99      1.00      0.99      1923
           1       0.95      0.89      0.92       136

    accuracy                           0.99      2059
   macro avg       0.97      0.94      0.96      2059
weighted avg       0.99      0.99      0.99      2059

[[1917    6]
 [  15  121]]


In [9]:
print("cm_staked:", cm_staked)
print("Shape:", cm_staked.shape)
print("NaNs:", np.isnan(cm_staked).any(), "Infs:", np.isinf(cm_staked).any())


cm_staked: [[1917    6]
 [  13  123]]
Shape: (2, 2)
NaNs: False Infs: False


In [ ]:
print(cm_staked)

plt.figure(figsize=(4, 4))
sns.heatmap(cm_staked, annot=True, fmt='d', cmap='Blues',
            xticklabels=["NavD", "AvD"],
            yticklabelsSSS=["NavD", "AvD"])
plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.title("Confusion Matrix – Stacked Model")
plt.tight_layout()
plt.show()

# Future work

Tune the base modeks with stacking in mind to maybe get slightly better results

- Train a slightly underfit RF and a slightly overfit NN to capture different biases
- Use different feature subsets for each model (a technique called stacked generalization with feature diversity)

# LIMITATION
We did a random split - the model might be able to somehow use temporal trends and therefore is really good in predicting 


Random split might be OKEY IF: Goal is to simulate short-term nowcasting, where the model is updated frequently and predicts tomorrow based on today’s data
→ Then a random split may approximate real-world model updating.

KEEP IN MIND
If your goal is to test generalization across seasons (e.g., will this model trained on 2018–2020 work in 2023?),
→ You must split by year or season to avoid data leakage across time.
!!! You don't randomly split in time-series datasets because it doesn't respect the temporal order and causes data-leakage, e.g. unintentionally inferring the trend of future samples !!!

### Problem that we might have:

Training set might include post-avalanche conditions for some days.
Your model then learns what happens after an avalanche — e.g., lower temperatures, melting, snowpack reset.
But in the real world, you don’t get to see the “after” until the event happens.

--> the model could be “cheating” by indirectly learning patterns that follow an avalanche — not just what leads to it.
It’s like trying to predict a car crash, but your model gets access to the scene of the crash, the police report, and the damaged car — all of which come after the crash.

You might want to:

- Try removing post-avalanche days from the training set for evaluation.
- Add an index buffer (e.g. don’t allow days within 2–3 days after an avalanche to appear in the training set).

# Correct approach to avoid this
Split by time (e.g., by season or year) — never randomly, unless you're doing short-term nowcasting.

- Filter out post-avalanche days from your training set (e.g., 2–3 days after each avalanche).

- This prevents learning "effects" instead of "causes."